# Training MNIST 🚀

In this colab we showcase how to train a diffusion model on MNIST dataset. This colab can run on any colab
backend.

In [ ]:
################################################################################
# Common modules
################################################################################

import functools
from etils import ecolab
import flax.linen as nn
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np
import optax
import tensorflow_datasets as tfds
import tqdm

################################################################################
# Hackable diffusion modules
################################################################################

with ecolab.adhoc(
    reload=["hackable_diffusion", "kauldron", "lark"],
    invalidate=False,
    cell_autoreload=True,
):
  from hackable_diffusion import hd

In [ ]:
diffusion_network = hd.diffusion_network
time_sampling = hd.training.time_sampling
gaussian = hd.corruption.gaussian
schedules = hd.corruption.schedules
conditioning_encoder = hd.architecture.conditioning_encoder
dit = hd.architecture.dit
dit_blocks = hd.architecture.dit_blocks
wrappers = hd.inference.wrappers
diffusion_inference = hd.inference.diffusion_inference
gaussian_loss = hd.training.gaussian_loss
time_scheduling = hd.sampling.time_scheduling
sampling = hd.sampling.sampling
gaussian_step_sampler = hd.sampling.gaussian_step_sampler

# Prepare MNIST data

Load all MNIST in memory.

MNIST data is $28 \times 28 \times 1$ (the images are scaled between $-1.0$ and $1.0$).

In [ ]:
ds = tfds.as_numpy(tfds.load('mnist', split='train', batch_size=-1))
all_data = ds['image'].astype(np.float32) / 127.5 - 1.0
all_data = all_data.reshape(-1, 28, 28, 1)
all_labels = ds['label'].astype(np.int32)

# Move to device once
all_data = jnp.array(all_data)
all_labels = jnp.array(all_labels)
dataset_size = all_data.shape[0]
print(f'Loaded {dataset_size} samples. Shape: {all_data.shape}')

In [ ]:
fig, axes = plt.subplots(8, 8, figsize=(8, 8))
for idx, ax in enumerate(axes.flatten()):
  ax.imshow(all_data[idx])
  ax.axis('off')
plt.tight_layout()
plt.show()

# Define all diffusion model modules

## Noise process

We use Rectified Flow noise schedule $x_{t} = (1-t) x_0 + t \epsilon$, $\epsilon \sim N(0, I)$

In [ ]:
schedule = schedules.RFSchedule()
process = gaussian.GaussianProcess(schedule=schedule)

Visualize noise process

In [ ]:
num_noises = 7
fig, axes = plt.subplots(
    ncols=num_noises, figsize=(num_noises * 4, 4), sharex=True, sharey=True
)

corrupt_rng = jax.random.PRNGKey(10)
idx = 0
for time in jnp.linspace(1e-3, 1.0 - 1e-3, num=num_noises):
  xt, _ = process.corrupt(
      key=corrupt_rng,
      x0=jnp.array(all_data[0]),
      time=jnp.ones((1,)) * time,
  )
  ax = axes[idx]
  ax.imshow(xt)
  ax.axis('off')
  ax.set_title(f'Time = {time}')
  idx += 1

## Define diffusion network backbone

First, we define diffusion backbone -- an architecture which takex `x` and `conditioning_embeddings`, as well as `is_training` and returns the same type as `x`.

Here, we use a small version of `DiT`.

In [ ]:
HIDDEN_DIM = 64
patch_size = (4, 4)
mlp_ratio = 4.0
num_blocks = 3
backbone = dit.DiT(
    num_blocks=num_blocks,
    block=dit_blocks.DiTBlockFlux(
        num_heads=4,
        hidden_size=HIDDEN_DIM,
        mlp_ratio=mlp_ratio,
    ),
    encoder=dit_blocks.Patchify(
        patch_size=patch_size, embedding_dim=HIDDEN_DIM
    ),
    decoder=dit_blocks.DePatchify(
        patch_size=patch_size, output_shape=(28, 28, 1)
    ),
    absolute_posenc=dit_blocks.PositionalEmbedding(),
)

## Define conditioning logic

Now, we define the conditioning embedders as well as the time encoder. The conditioning encoder processes each conditioning (in the case of MNIST data, each batch comes with its label (`label`)).

The conditioning encoder is a dictionary with key `label` (and here the value is a `nn.Module` which is given by a simple `LabelEmbedding` module). If you want to train a purely unconditional model, set `conditioning_embedders = {}`.



In [ ]:
################################################################################
# Conditional diffusion.
################################################################################

conditioning_embedders = {
    'label': conditioning_encoder.LabelEmbedder(
        num_classes=10,
        num_features=HIDDEN_DIM,
        conditioning_key='label',
    )
}

encoder = conditioning_encoder.ConditioningEncoder(
    time_embedder=conditioning_encoder.SinusoidalTimeEmbedder(
        activation='gelu', embedding_dim=HIDDEN_DIM, num_features=HIDDEN_DIM
    ),
    conditioning_embedders=conditioning_embedders,
    merge_embeddings_fn=conditioning_encoder.ConcatEmbeddings(),
    conditioning_rules={
        'time': 'adaptive_norm',
        'label': 'adaptive_norm',
    },
)

## Putting all together into diffusion network

In [ ]:
network = diffusion_network.DiffusionNetwork(
    backbone_network=backbone,
    conditioning_encoder=encoder,
    prediction_type='x0',
)

Model visualization

In [ ]:
summary_depth = 2  # @param {type: "integer"}

tabulate_fn = nn.tabulate(
    network,
    jax.random.PRNGKey(42),
    depth=summary_depth,
    console_kwargs={"force_jupyter": True, "soft_wrap": True},
)

dummy_time = jnp.ones((1,))
dummy_xt = jnp.ones((1, 28, 28, 1))
dummy_conditioning = {"label": jnp.ones((1,)).astype(jnp.int32)}

print(
    tabulate_fn(
        dummy_time,
        dummy_xt,
        dummy_conditioning,
        is_training=False,
    )
)

## Define time sampler, optimizer and loss function

The time is sampled uniformly in the interval $[\epsilon,1 - \epsilon]$.

The loss is simply the $\ell_2$ loss.

In [ ]:
time_sampler = time_sampling.UniformTimeSampler(
    span=hd.jax_helpers.SafeSpan(safety_epsilon=1e-3),
)

optimizer = optax.chain(
    optax.clip_by_global_norm(max_norm=1.0),
    optax.adamw(learning_rate=7e-3, b1=0.9, b2=0.99, weight_decay=0.0),
)

loss_fn = gaussian_loss.NoWeightGaussianLoss(prediction_type='x0')

## Define the parameters loss function and gradient function

Here we define the loss function as well as gradient function to be dependent on NN parameters. This is needed for training the neural network.

In [ ]:
@jax.jit
def params_loss_fn(params, x0, conditioning, rng):
  time_rng, corrupt_rng = jax.random.split(rng, 2)
  time = time_sampler(key=time_rng, data_spec=x0)
  xt, targets = process.corrupt(key=corrupt_rng, x0=x0, time=time)
  output = network.apply(
      {'params': params},
      time=time,
      xt=xt,
      conditioning=conditioning,
      is_training=True,
      rngs={'dropout': rng},
  )
  out = jnp.mean(loss_fn(preds=output, targets=targets, time=time))
  return out, {'loss': out}


grad_fn = jax.jit(jax.grad(params_loss_fn, has_aux=True))

Wrapping the whole update into `update_fn` since it makes the updates much faster

In [ ]:
@jax.jit
def update_fn(params, opt_state, x0, conditioning, rng):
  grads, metrics = grad_fn(params, x0, conditioning, rng)
  updates, opt_state = optimizer.update(grads, opt_state, params=params)
  params = optax.apply_updates(params, updates)
  return params, opt_state, metrics

## Train the model

Training the model should take less than 10 minutes.

In [ ]:
nepochs = 50
batch_size = 1024
steps_per_epoch = dataset_size // batch_size

In [ ]:
rng = jax.random.PRNGKey(0)

params = network.initialize_variables(
    input_shape=(1, 28, 28, 1),
    conditioning_shape={'label': (1,)},
    key=rng,
    is_training=True,
)['params']

opt_state = optimizer.init(params)

ema_decay = 0.999
ema_params = params

losses = []
for epoch in tqdm.tqdm(range(1, nepochs + 1)):
  # Shuffle indices each epoch
  rng, shuffle_rng = jax.random.split(rng)
  perm = jax.random.permutation(shuffle_rng, dataset_size)

  epoch_loss = 0.0
  for i in range(steps_per_epoch):
    idx = perm[i * batch_size : (i + 1) * batch_size]
    x0 = all_data[idx]
    conditioning = {'label': all_labels[idx]}

    rng, step_rng = jax.random.split(rng)
    params, opt_state, metrics = update_fn(
        params, opt_state, x0, conditioning, step_rng
    )
    ema_params = jax.tree.map(
        lambda ema, p: ema_decay * ema + (1 - ema_decay) * p, ema_params, params
    )
    epoch_loss += metrics['loss']
  if epoch % 10 == 0:
    print(f'Epoch = {epoch}, Avg loss = {epoch_loss / steps_per_epoch:.4f}')
  losses.append(epoch_loss / steps_per_epoch)

In [ ]:
plt.plot(losses)

## It's inference time

Below, we define the inference function.
It creates a pure jax function which takes `t`, `xt` and `c` to return the expected value of `x0`.

In [ ]:
base_inference_fn = wrappers.FlaxLinenInferenceFn(
    network=network,
    params=ema_params,
)
inference_fn = diffusion_inference.GuidedDiffusionInferenceFn(
    base_inference_fn=base_inference_fn
)

## Sampler -- time_schedule, stepper and sampler itself

In [ ]:
num_sampling_steps = 100  # Number of denoising steps
stochasticity_level = 1.0  # Stochasticity coefficient in DDIM

time_schedule = time_scheduling.UniformTimeSchedule(
    span=hd.jax_helpers.SafeSpan(safety_epsilon=1e-3),
)
stepper = gaussian_step_sampler.DDIMStep(
    corruption_process=process, stoch_coeff=stochasticity_level
)

sampler = sampling.DiffusionSampler(
    time_schedule=time_schedule, stepper=stepper, num_steps=num_sampling_steps
)
sampler = functools.partial(sampler, inference_fn=inference_fn)
sampler = jax.jit(jax.experimental.checkify.checkify(sampler))

## Sampling the data

* First, we sample the data taking the conditioning from a batch of data, allowing to approximate $p(x_0)$

* Second, we sample data with a given label, allowing to sample $p(x_0 | c)$

In [ ]:
eval_seed = 15
num_samples = 64
data_spec = jnp.ones((num_samples, 28, 28, 1), dtype=jnp.float32)
specific_label = 5

rng, shuffle_rng = jax.random.split(jax.random.PRNGKey(eval_seed))
perm = jax.random.permutation(shuffle_rng, dataset_size)

idx = perm[i * num_samples : (i + 1) * num_samples]
eval_x0 = all_data[idx]
eval_conditioning = {"label": all_labels[idx]}

################################################################################
# Sample conditionally using dataset
################################################################################

key = jax.random.PRNGKey(0)
initial_noise = process.sample_from_invariant(key=key, data_spec=data_spec)
_, (out_cond, _) = sampler(
    rng=key, initial_noise=initial_noise, conditioning=eval_conditioning
)

################################################################################
# Sample from a given label
################################################################################

key = jax.random.PRNGKey(1)
initial_noise = process.sample_from_invariant(key=key, data_spec=data_spec)
conditioning = {
    "label": jnp.ones((num_samples,)).astype(jnp.int32) * specific_label
}
_, (out_label, _) = sampler(
    rng=key, initial_noise=initial_noise, conditioning=conditioning
)

Visualize true dataset

In [ ]:
cur_mnist_plot_images = eval_x0
fig, axes = plt.subplots(8, 8, figsize=(8, 8))
for img, ax in zip(cur_mnist_plot_images[:64], axes.flatten()):
  ax.imshow(img)
  ax.axis('off')

plt.tight_layout()
plt.show()

Visualize samples from $p(x_0)$

In [ ]:
cur_mnist_plot_images = out_cond.xt
fig, axes = plt.subplots(8, 8, figsize=(8, 8))
for img, ax in zip(out_cond.xt[:64].clip(-1.0, 1.0), axes.flatten()):
  ax.imshow(img.squeeze(), vmin=-1, vmax=1)
  ax.axis('off')
plt.tight_layout()
plt.show()

Visualize samples from $p(x_0 | c)$

In [ ]:
cur_mnist_plot_images = out_label.xt
fig, axes = plt.subplots(8, 8, figsize=(8, 8))
for img, ax in zip(cur_mnist_plot_images[:64].clip(-1.0, 1), axes.flatten()):
  ax.imshow(img)
  ax.axis('off')

plt.tight_layout()
plt.show()